### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

## initialize the chat model for the tool: here any Groq model is used
model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. They are birds, right? Some species can mimic human speech, like parakeets, macaws, and cockatiels. But why do they do that? Is it just imitation, or is there a deeper reason?\n\nFirst, maybe it\'s about communication. In the wild, parrots use a variety of sounds to interact with each other. They might scream to defend their territory, call to their mates, or warn others in their flock. If they\'re in captivity, maybe they imitate human speech because they\'re trying to communicate with their human caregivers. But how do they learn that?\n\nI remember reading that some birds have a part of their brain called the song system, which is involved in vocal learning. Parrots might use this to learn sounds from their environment. So maybe talking is just an extension of their natural ability to mimic sounds. But why specifically human speech? Is it because they\'re 

In [2]:
from langchain.tools import tool

## tool is used as a decorator here
@tool
def get_weather(location:str)->str:
    ## Docstring is pretty important
    ## It enables the LLM to specify after binding
    """Get the weather at a location"""
    return f"It's sunny in {location}"

## bind the tool with LLM using bind_tools function
model_with_tools = model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which in this case is Boston. I\'ll call the function with "Boston" as the location. Make sure the JSON is correctly formatted with the name and arguments. No other functions are available, so this should be straightforward.\n', 'tool_calls': [{'id': 'skgzdx5rs', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 154, 'total_tokens': 256, 'completion_time': 0.168648262, 'completion_tokens_details': {'reasoning_tokens': 78}, 'prompt_time': 0.006335735, 'prompt_tokens_details': None, 'queue_time': 0.050765783, 'total_time': 0.174983997}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_99d722e776', 'service_tier': 'on_demand', 'finish_reason': 'tool_cal

In [6]:
for tool_call in response.tool_calls:
    ## View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


### Tool Execution Loops

In [ ]:
## Step 1: Model generates tool calls
messages = [{
    "role": "user", "content": "What's the weather in Boston?"
}]

ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

In [ ]:
## Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

## Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston is sunny.


In [9]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call this function with "Boston" as the location. I\'ll make sure to format the tool call correctly within the XML tags as instructed.\n', 'tool_calls': [{'id': '5p75n907g', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 153, 'total_tokens': 246, 'completion_time': 0.135158631, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.007405967, 'prompt_tokens_details': None, 'queue_time': 0.091246139, 'total_time': 0.142564598}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_deman